# Current period-arbitration audit on real ASAS-SN candidates

This notebook makes the complete period-selection process inspectable on a balanced set of healthy real light curves. It deliberately includes objects **with a clean external catalog period** and objects for which **none of the integrated external catalogs supplies a period**.

It answers five separate questions:

1. Which periods do the current global search functions propose?
2. How do merging, harmonic expansion, fixed-period scoring, and local refinement change the candidate pool?
3. Why does the deterministic arbitration score prefer its top period over the alternatives?
4. When an external period exists, does the selected period match its exact convention or only its harmonic family?
5. When no external period exists, how strong is the internal evidence (rank margin, method support, repeatability, and agreement with the stored production period)?

The fresh calculation uses `PeriodCandidateMethodsConfig()` from the **current checkout without changing its defaults**. It saves resumable per-object checkpoints, aggregate Parquet/CSV tables, and every figure under a versioned output directory. It does not write to `review.db` or change any production period.


In [ ]:
from __future__ import annotations

from dataclasses import asdict
from datetime import datetime, timezone
import hashlib
from importlib import metadata as importlib_metadata
import json
import os
from pathlib import Path
import subprocess

for _name in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "NUMBA_NUM_THREADS",
):
    os.environ.setdefault(_name, "1")

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed, parallel_config

from malca.evaluation.period_arbitration_audit import (
    annotate_external_period_groups,
    choose_case_studies,
    collect_candidate_audits,
    completed_audit_source_ids,
    current_default_config_table,
    load_prepared_light_curve,
    run_current_deterministic_arbitration_safe,
    save_candidate_audit,
    select_healthy_period_cohort,
    topk_catalog_recovery,
    utility_family_table,
    write_aggregate_audits,
)
from malca.evaluation.period_candidate_methods import PeriodCandidateMethodsConfig
from malca.evaluation.period_cost_accuracy import (
    add_catalog_reference,
    evaluate_stored_strategies,
    load_review_period_snapshot,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180})


## Run configuration and persistent outputs

The default cohort contains 12 light curves from each reference group (24 total) and uses 12 processes. On the completed M5 Max benchmark, a comparable V3 calculation took roughly tens of seconds per source before parallelism. Increase the cohort only after this default run completes.

The output tag is part of the provenance. Reusing a tag with a different configuration is rejected rather than silently combining incompatible checkpoints.


In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "malca").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the MALCA checkout")


ROOT = find_repo_root(Path.cwd().resolve())
RUN_ROOT = ROOT / "output/runs/dat3-full-extended_2026-07-01-v4"
REVIEW_DB = RUN_ROOT / "review/review.db"

AUDIT_TAG = os.environ.get("MALCA_PERIOD_AUDIT_TAG", "current_defaults_20260803")
if Path(AUDIT_TAG).name != AUDIT_TAG:
    raise ValueError("MALCA_PERIOD_AUDIT_TAG must be one path-safe name")
OUTPUT_ROOT = ROOT / "output/evaluation/period_arbitration_real_audit" / AUDIT_TAG
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

N_PER_GROUP = int(os.environ.get("MALCA_PERIOD_AUDIT_N_PER_GROUP", "12"))
WORKERS = int(os.environ.get("MALCA_PERIOD_AUDIT_WORKERS", "12"))
INNER_THREADS = int(os.environ.get("MALCA_PERIOD_AUDIT_INNER_THREADS", "1"))
BATCH_SIZE = int(os.environ.get("MALCA_PERIOD_AUDIT_BATCH_SIZE", str(max(1, WORKERS))))
COHORT_SEED = int(os.environ.get("MALCA_PERIOD_AUDIT_SEED", "20260803"))
MATCH_TOLERANCE = float(os.environ.get("MALCA_PERIOD_AUDIT_MATCH_TOLERANCE", "0.05"))
CASES_PER_GROUP = int(os.environ.get("MALCA_PERIOD_AUDIT_CASES_PER_GROUP", "4"))
RUN_FRESH = os.environ.get("MALCA_PERIOD_AUDIT_RUN_FRESH", "1") == "1"

CONFIG = PeriodCandidateMethodsConfig()
CONFIG_TABLE = current_default_config_table(CONFIG)
CONFIG_TABLE.to_parquet(OUTPUT_ROOT / "current_default_config.parquet", index=False)

git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True
).strip()
source_paths = [
    ROOT / "malca/evaluation/period_arbitration_audit.py",
    ROOT / "malca/evaluation/period_candidate_methods.py",
    ROOT / "malca/evaluation/period_candidate_ranker.py",
    ROOT / "malca/notebooks/evaluation/period_arbitration_real_candidate_audit.ipynb",
]
source_sha256 = {
    str(path.relative_to(ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in source_paths
}
git_dirty = bool(
    subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip()
)
manifest_payload = {
    "schema_version": "period-arbitration-real-audit-v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_commit,
    "git_dirty": git_dirty,
    "source_sha256": source_sha256,
    "audit_tag": AUDIT_TAG,
    "cohort_seed": COHORT_SEED,
    "n_per_group": N_PER_GROUP,
    "match_tolerance": MATCH_TOLERANCE,
    "config": asdict(CONFIG),
    "dependencies": {
        name: importlib_metadata.version(name)
        for name in ("astropy", "numpy", "pandas", "scipy", "supersmoother")
    },
}
fingerprint_input = dict(manifest_payload)
fingerprint_input.pop("created_utc")
manifest_payload["fingerprint"] = hashlib.sha256(
    json.dumps(fingerprint_input, sort_keys=True, default=str).encode()
).hexdigest()
manifest_path = OUTPUT_ROOT / "run_manifest.json"
if manifest_path.is_file():
    existing_manifest = json.loads(manifest_path.read_text())
    if existing_manifest.get("fingerprint") != manifest_payload["fingerprint"]:
        raise RuntimeError(
            "This output tag already contains a different configuration. "
            "Choose a new MALCA_PERIOD_AUDIT_TAG."
        )
else:
    manifest_path.write_text(json.dumps(manifest_payload, indent=2, sort_keys=True) + "\n")

print({
    "root": str(ROOT),
    "review_db": str(REVIEW_DB),
    "output_root": str(OUTPUT_ROOT),
    "git_commit": git_commit,
    "workers": WORKERS,
    "n_per_group": N_PER_GROUP,
    "run_fresh": RUN_FRESH,
    "pdm_method": CONFIG.pdm_method,
    "max_scored_candidates": CONFIG.max_scored_candidates,
    "max_refined_candidates": CONFIG.max_refined_candidates,
})


## What the current functions do

The audit executes this sequence for every light curve:

**prepared light curve → global method proposals → frequency-space merging → harmonic expansion → diverse scoring shortlist → fixed-period diagnostics → LS/BLS local refinement → deterministic percentile arbitration**

The table below is generated directly from the current configuration class. It is therefore the authoritative description of today's zero-argument defaults, including the currently selected PDM implementation, candidate limits, grids, and refinement settings.


In [ ]:
important_parameters = [
    "enabled_global_methods",
    "enabled_fixed_methods",
    "pdm_method",
    "short_min_period_days",
    "short_max_period_days",
    "long_min_period_days",
    "long_max_baseline_fraction",
    "long_absolute_max_period_days",
    "harmonic_factors",
    "max_scored_candidates",
    "max_refined_candidates",
    "top_k_ls",
    "top_k_general",
    "top_k_event",
    "ls_samples_per_peak",
    "pdm_n_frequency",
    "pdm_n_phase_bins",
    "pdm_plavchan_phase_width",
    "ce_n_frequency",
    "ce_n_phase_bins",
    "ce_n_mag_bins",
    "bls_n_frequency",
    "bls_duration_fractions",
    "bls_oversample",
    "fixed_ls_refine_n_frequency",
    "fixed_bls_max_refinement_seeds",
]
display(
    CONFIG_TABLE.loc[CONFIG_TABLE["parameter"].isin(important_parameters)]
    .set_index("parameter")
    .loc[important_parameters]
)

method_inventory = pd.DataFrame({
    "global proposer": list(CONFIG.enabled_global_methods),
})
display(method_inventory)
print("Fixed-period diagnostics:", ", ".join(CONFIG.enabled_fixed_methods))


## Population accounting and external-reference definitions

A clean external reference is built by MALCA's existing catalog-consensus code and is withheld from fresh candidate generation and ranking. The `no_external_period` group is stricter than “no clean consensus”: none of the integrated Gaia EB, VSX, ASAS-SN variable, ZTF, or OGLE period fields may contain a finite positive period. Objects with conflicting or otherwise unclean external information are counted but excluded from the balanced case cohort.

External agreement is a validation layer, not absolute truth. Exact agreement uses a 5% relative tolerance. Harmonic-family agreement additionally allows factors 1/4, 1/3, 1/2, 2, 3, and 4.


In [ ]:
real_periods = add_catalog_reference(load_review_period_snapshot(REVIEW_DB))
real_periods = annotate_external_period_groups(real_periods)
real_periods["lc_exists"] = real_periods["lc_path"].map(
    lambda value: Path(str(value)).is_file() if pd.notna(value) else False
)

population_accounting = (
    real_periods.groupby("external_period_group", dropna=False)
    .agg(
        sources=("candidate_id", "size"),
        light_curve_exists=("lc_exists", "sum"),
        median_points=("n_points", "median"),
        median_magnitude=("median_mag", "median"),
    )
    .reset_index()
)
population_accounting.to_parquet(OUTPUT_ROOT / "population_accounting.parquet", index=False)
display(population_accounting)

stored_strategy_summary = evaluate_stored_strategies(
    real_periods,
    tolerance=MATCH_TOLERANCE,
)
stored_strategy_summary.to_parquet(
    OUTPUT_ROOT / "stored_strategy_summary.parquet", index=False
)
display(stored_strategy_summary[[
    "strategy",
    "coverage_all",
    "exact_agreement_conditional",
    "family_agreement_conditional",
]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
population_plot = population_accounting.sort_values("sources", ascending=True)
axes[0].barh(population_plot["external_period_group"], population_plot["sources"], color="#4c78a8")
axes[0].set_xlabel("Number of candidates")
axes[0].set_title("External-period availability in Review")
axes[0].grid(axis="x", alpha=0.2)

independent = stored_strategy_summary.loc[stored_strategy_summary["reference_independent"]].copy()
independent = independent.sort_values("family_agreement_conditional")
axes[1].barh(independent["label"], independent["family_agreement_conditional"], color="#59a14f")
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Harmonic-family agreement")
axes[1].set_title("Stored internally computed periods")
axes[1].grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "population_and_stored_periods.png", bbox_inches="tight")
plt.show()


## Balanced healthy cohort

“Healthy” means that the light-curve file exists, its median magnitude is between 12 and 15, and at least 100 input measurements are recorded. Sampling is balanced across one-magnitude bins and lower/higher observation-count strata within each external-reference group.

This is a diagnostic cohort, not a population-weighted accuracy sample. Its purpose is to expose successes, harmonic-convention differences, and ambiguous cases in enough detail to understand the ranker's behavior.


In [ ]:
cohort = select_healthy_period_cohort(
    real_periods,
    n_per_group=N_PER_GROUP,
    seed=COHORT_SEED,
    magnitude_min=12.0,
    magnitude_max=15.0,
    min_points=100,
)
cohort.to_parquet(OUTPUT_ROOT / "cohort.parquet", index=False)
display(cohort[[
    "candidate_id",
    "cohort_group",
    "sample_stratum",
    "median_mag",
    "n_points",
    "catalog_reference_period",
    "periodicity_period",
]])

GROUP_COLORS = {
    "external_reference": "#4c78a8",
    "no_external_period": "#f28e2b",
}
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
for group_name, group in cohort.groupby("cohort_group", sort=True):
    color = GROUP_COLORS[group_name]
    axes[0].hist(group["median_mag"], bins=np.linspace(12, 15, 10), alpha=0.55, label=group_name, color=color)
    axes[1].hist(group["n_points"], bins=10, alpha=0.55, label=group_name, color=color)
axes[0].set(xlabel="Median magnitude", ylabel="Candidates", title="Magnitude balance")
axes[1].set(xlabel="Input measurements", ylabel="Candidates", title="Cadence-density balance")
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "cohort_balance.png", bbox_inches="tight")
plt.show()


## Fresh current-default arbitration

This is the expensive cell. It uses process-level parallelism and one numerical thread per worker. Every completed object is written to its own checkpoint directory, so an interrupted run resumes without recomputing finished sources.

Set `MALCA_PERIOD_AUDIT_RUN_FRESH=0` to inspect an already completed tag without launching new searches.


In [ ]:
cohort_ids = set(cohort["candidate_id"].astype(str))
completed = completed_audit_source_ids(CHECKPOINT_ROOT)
pending_records = [
    dict(row)
    for _, row in cohort.iterrows()
    if str(row["candidate_id"]) not in completed
]

if RUN_FRESH:
    print(f"Completed checkpoints: {len(completed & cohort_ids)}/{len(cohort)}; pending: {len(pending_records)}")
    for start in range(0, len(pending_records), BATCH_SIZE):
        batch = pending_records[start:start + BATCH_SIZE]
        with parallel_config(
            backend="loky",
            n_jobs=WORKERS,
            inner_max_num_threads=INNER_THREADS,
        ):
            results = Parallel(batch_size=1, pre_dispatch="2*n_jobs")(
                delayed(run_current_deterministic_arbitration_safe)(
                    record,
                    config=CONFIG,
                    match_tolerance=MATCH_TOLERANCE,
                )
                for record in batch
            )
        for result in results:
            save_candidate_audit(result, checkpoint_root=CHECKPOINT_ROOT)
        print(f"Checkpointed {min(start + len(batch), len(pending_records))}/{len(pending_records)} pending sources")
else:
    print("Fresh execution disabled; loading existing checkpoints.")

audits = collect_candidate_audits(CHECKPOINT_ROOT)
id_columns = {
    "summary": "candidate_id",
    "candidate_scores": "source_id",
    "proposals": "source_id",
    "stage_counts": "candidate_id",
    "search_status": "source_id",
}
for name, frame in list(audits.items()):
    id_column = id_columns[name]
    if not frame.empty and id_column in frame:
        audits[name] = frame.loc[frame[id_column].astype(str).isin(cohort_ids)].copy()
write_aggregate_audits(audits, output_root=OUTPUT_ROOT)

summaries = audits["summary"]
candidate_scores = audits["candidate_scores"]
proposals = audits["proposals"]
stage_counts = audits["stage_counts"]
search_status = audits["search_status"]

status_accounting = summaries.groupby("status", dropna=False).size().rename("sources").reset_index()
display(status_accounting)
if not summaries.loc[summaries["status"].ne("ok")].empty:
    display(summaries.loc[summaries["status"].ne("ok"), ["candidate_id", "error"]])


## Best periods selected for the cohort

`selected_period_days` is the top-ranked period returned by the fresh deterministic arbitration. The runner-up and score margin show how decisively it won. External periods are displayed only after selection; rows without an external period remain valid outputs but have no direct accuracy label.


In [ ]:
ok = summaries.loc[summaries["status"].eq("ok")].copy()
best_period_columns = [
    "candidate_id",
    "cohort_group",
    "selected_period_days",
    "runner_up_period_days",
    "stored_period_days",
    "catalog_reference_period",
    "selected_exact_match",
    "selected_family_match",
    "selected_score",
    "score_margin",
    "selected_independent_method_family_count",
    "selected_candidate_stage",
    "selected_contributing_methods",
    "total_seconds",
]
best_periods = ok[best_period_columns].sort_values(["cohort_group", "candidate_id"])
best_periods.to_parquet(OUTPUT_ROOT / "best_periods.parquet", index=False)
best_periods.to_csv(OUTPUT_ROOT / "best_periods.csv", index=False)
display(best_periods)

group_summary = (
    ok.groupby("cohort_group", dropna=False)
    .agg(
        sources=("candidate_id", "size"),
        median_runtime_s=("total_seconds", "median"),
        median_score=("selected_score", "median"),
        median_margin=("score_margin", "median"),
        median_method_families=("selected_independent_method_family_count", "median"),
        exact_catalog_agreement=("selected_exact_match", "mean"),
        family_catalog_agreement=("selected_family_match", "mean"),
        agreement_with_stored_family=("selected_vs_stored_family", "mean"),
    )
    .reset_index()
)
group_summary.to_parquet(OUTPUT_ROOT / "cohort_group_summary.parquet", index=False)
display(group_summary)


## Candidate funnel and computational cost

The funnel distinguishes the complete harmonic-expanded proposal bank from the expensive fixed-scoring shortlist and from newly generated local refinements. The final pool contains both the scored shortlist and the retained refinements.


In [ ]:
stage_summary = (
    stage_counts.groupby(["cohort_group", "stage", "stage_order"], as_index=False)
    .agg(median_candidates=("candidate_count", "median"), minimum=("candidate_count", "min"), maximum=("candidate_count", "max"))
    .sort_values("stage_order")
)
display(stage_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for group_name, group in stage_summary.groupby("cohort_group", sort=True):
    axes[0].plot(
        group["stage_order"],
        group["median_candidates"],
        marker="o",
        linewidth=2,
        label=group_name,
        color=GROUP_COLORS[group_name],
    )
    axes[0].fill_between(
        group["stage_order"], group["minimum"], group["maximum"],
        alpha=0.12, color=GROUP_COLORS[group_name],
    )
stage_labels = stage_summary.drop_duplicates("stage_order").sort_values("stage_order")
axes[0].set_xticks(stage_labels["stage_order"], stage_labels["stage"], rotation=30, ha="right")
axes[0].set_ylabel("Candidates per source")
axes[0].set_title("Candidate-pool funnel (median and range)")
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.2)

runtime_columns = [
    "preparation_seconds",
    "global_search_seconds",
    "candidate_bank_seconds",
    "fixed_scoring_seconds",
    "refinement_seconds",
    "ranking_seconds",
]
runtime_labels = ["prepare", "global searches", "bank", "fixed scoring", "refine + rescore", "rank"]
runtime_medians = ok[runtime_columns].median()
axes[1].barh(runtime_labels, runtime_medians.values, color="#9c755f")
axes[1].set_xscale("log")
axes[1].set_xlabel("Median seconds per source (log scale)")
axes[1].set_title("Where computation is spent")
axes[1].grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "candidate_funnel_and_runtime.png", bbox_inches="tight")
plt.show()


## External-catalog performance: candidate ceiling versus selected period

Candidate-oracle recovery asks whether a matching candidate existed anywhere in the generated pool. Top-1 recovery asks whether arbitration actually selected it. The separation between those curves is selection error; failure of both is candidate-generation error.


In [ ]:
known = ok.loc[ok["cohort_group"].eq("external_reference")].copy()
topk = topk_catalog_recovery(candidate_scores)
topk.to_parquet(OUTPUT_ROOT / "topk_catalog_recovery.parquet", index=False)
display(topk)

recovery_values = pd.Series({
    "selected exact": pd.to_numeric(known["selected_exact_match"], errors="coerce").mean(),
    "selected family": pd.to_numeric(known["selected_family_match"], errors="coerce").mean(),
    "ranked-pool exact": pd.to_numeric(known["ranked_pool_oracle_exact"], errors="coerce").mean(),
    "ranked-pool family": pd.to_numeric(known["ranked_pool_oracle_family"], errors="coerce").mean(),
    "all-candidate exact": pd.to_numeric(known["all_candidate_oracle_exact"], errors="coerce").mean(),
    "all-candidate family": pd.to_numeric(known["all_candidate_oracle_family"], errors="coerce").mean(),
})

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))
colors = ["#4c78a8", "#59a14f", "#4c78a8", "#59a14f", "#4c78a8", "#59a14f"]
axes[0].barh(recovery_values.index, recovery_values.values, color=colors)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Fraction of external-reference cohort")
axes[0].set_title("Generation ceiling versus adopted period")
axes[0].grid(axis="x", alpha=0.2)
for y, value in enumerate(recovery_values.values):
    axes[0].text(min(value + 0.02, 0.96), y, f"{value:.2f}", va="center")

if not topk.empty:
    axes[1].plot(topk["top_k"], topk["exact_recovery"], marker="o", label="exact", color="#4c78a8")
    axes[1].plot(topk["top_k"], topk["family_recovery"], marker="o", label="harmonic family", color="#59a14f")
    axes[1].set_xscale("log", base=2)
    axes[1].set_ylim(0, 1.02)
    axes[1].set_xlabel("Candidates inspected, K")
    axes[1].set_ylabel("Reference recovered within top K")
    axes[1].set_title("Ranker top-K saturation")
    axes[1].legend(frameon=False)
    axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "catalog_recovery_and_topk.png", bbox_inches="tight")
plt.show()


## Selected-period convention relative to the external catalogs

Points on the one-to-one line use the same convention as the reference catalog. Parallel harmonic lines reveal otherwise coherent half-, double-, and higher-multiple solutions. These should not automatically be interpreted as physically wrong periods.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0))
if not known.empty:
    x = pd.to_numeric(known["catalog_reference_period"], errors="coerce")
    y = pd.to_numeric(known["selected_period_days"], errors="coerce")
    exact_mask = known["selected_exact_match"].eq(True)
    family_mask = known["selected_family_match"].eq(True) & ~exact_mask
    miss_mask = ~known["selected_family_match"].eq(True)
    axes[0].scatter(x[exact_mask], y[exact_mask], label="exact", color="#59a14f", s=45)
    axes[0].scatter(x[family_mask], y[family_mask], label="harmonic only", color="#f28e2b", s=45)
    axes[0].scatter(x[miss_mask], y[miss_mask], label="family mismatch", color="#e15759", s=48, marker="x")
    limits = [max(0.08, min(x.min(), y.min()) * 0.8), max(x.max(), y.max()) * 1.25]
    grid = np.geomspace(limits[0], limits[1], 300)
    for factor in (0.5, 1.0, 2.0):
        axes[0].plot(grid, factor * grid, linestyle="--" if factor != 1 else "-", linewidth=1, color="0.45", alpha=0.7)
        axes[0].text(grid[-1], factor * grid[-1], f" {factor:g}x", color="0.35", va="center")
    axes[0].set(xscale="log", yscale="log", xlim=limits, ylim=limits, xlabel="External reference period (d)", ylabel="Selected period (d)")
    axes[0].set_title("Selected versus external reference")
    axes[0].legend(frameon=False)
    axes[0].grid(alpha=0.2, which="both")

    ratio = y / x
    axes[1].hist(np.log10(ratio), bins=18, color="#4c78a8", alpha=0.8)
    for factor in (0.25, 1/3, 0.5, 1, 2, 3, 4):
        axes[1].axvline(np.log10(factor), color="0.4", linewidth=1, alpha=0.6)
        axes[1].text(np.log10(factor), axes[1].get_ylim()[1] * 0.96, f"{factor:g}x", rotation=90, va="top", ha="right", fontsize=8)
    axes[1].set_xlabel(r"$\log_{10}(P_{\rm selected}/P_{\rm catalog})$")
    axes[1].set_ylabel("Candidates")
    axes[1].set_title("Which period convention was selected?")
    axes[1].grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "selected_vs_catalog.png", bbox_inches="tight")
plt.show()


## Evidence when no external period is available

There is no accuracy label for this group. Instead, the plots show independent warning or support signals: the gap between the top two arbitration scores, the number of contributing method families, cycle-to-cycle template repeatability, and harmonic-family agreement with the previously stored production period. None of these quantities alone proves that a period is correct.


In [ ]:
unknown = ok.loc[ok["cohort_group"].eq("no_external_period")].copy()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

groups_for_box = [
    ok.loc[ok["cohort_group"].eq(group_name), "score_margin"].dropna().to_numpy()
    for group_name in ("external_reference", "no_external_period")
]
axes[0, 0].boxplot(groups_for_box, tick_labels=["external ref", "no external"], showfliers=True)
axes[0, 0].set_ylabel("Top score minus runner-up")
axes[0, 0].set_title("Arbitration separation")
axes[0, 0].grid(axis="y", alpha=0.2)

axes[0, 1].scatter(
    unknown["selected_independent_method_family_count"],
    unknown["score_margin"],
    c=unknown["selected_template_q"],
    cmap="viridis_r",
    s=65,
    edgecolor="0.2",
    linewidth=0.4,
)
axes[0, 1].set(xlabel="Independent proposing-method families", ylabel="Score margin", title="Internal support without a catalog")
axes[0, 1].grid(alpha=0.2)

q_values = pd.to_numeric(unknown["selected_template_q"], errors="coerce")
axes[1, 0].hist(q_values.dropna(), bins=10, color="#f28e2b", alpha=0.8)
axes[1, 0].set(xlabel="Held-out phase-template Q (lower is better)", ylabel="Candidates", title="Cycle-to-cycle repeatability")
axes[1, 0].grid(axis="y", alpha=0.2)

stored_family = pd.to_numeric(unknown["selected_vs_stored_family"], errors="coerce")
agreement = pd.Series({
    "same harmonic family": stored_family.mean(),
    "different / unavailable": 1.0 - stored_family.mean() if stored_family.notna().any() else np.nan,
})
axes[1, 1].bar(agreement.index, agreement.values, color=["#59a14f", "#bab0ab"])
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_ylabel("Fraction")
axes[1, 1].set_title("Fresh selection versus stored production period")
axes[1, 1].tick_params(axis="x", rotation=15)
axes[1, 1].grid(axis="y", alpha=0.2)

fig.tight_layout()
fig.savefig(FIGURE_ROOT / "no_external_internal_evidence.png", bbox_inches="tight")
plt.show()


## Which methods supplied the candidates that won?

A candidate can preserve provenance from several proposers after frequency-space merging. The first panel counts all raw proposals; the second counts every method contributing to the ultimately selected candidate. Current defaults omit the redundant AoV proposer, so the multiharmonic Fourier evidence appears only through the corresponding Lomb--Scargle searches.


In [ ]:
proposal_counts = proposals.groupby("search_method").size().sort_values()
selected_support_rows = []
for _, row in ok.iterrows():
    value = row["selected_contributing_methods"]
    try:
        methods = json.loads(value) if isinstance(value, str) else list(value)
    except (TypeError, json.JSONDecodeError):
        methods = [str(value)]
    for method in methods:
        selected_support_rows.append({"candidate_id": row["candidate_id"], "method": method})
selected_support = pd.DataFrame(selected_support_rows)
selected_counts = selected_support.groupby("method").size().sort_values() if not selected_support.empty else pd.Series(dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].barh(proposal_counts.index, proposal_counts.values, color="#4c78a8")
axes[0].set(xlabel="Raw candidates across cohort", title="Global proposal volume")
axes[0].grid(axis="x", alpha=0.2)
axes[1].barh(selected_counts.index, selected_counts.values, color="#59a14f")
axes[1].set(xlabel="Selected candidates receiving support", title="Provenance of adopted periods")
axes[1].grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "proposal_and_winner_methods.png", bbox_inches="tight")
plt.show()

search_health = (
    search_status.groupby(["method", "status"]).size().rename("sources").reset_index()
)
search_health.to_parquet(OUTPUT_ROOT / "global_search_health.parquet", index=False)
display(search_health)


## Score anatomy: what separated the winner from the runner-up?

The deterministic ranker converts each diagnostic into a within-light-curve utility from 0 to 1 and averages all included utilities with equal weight. For readability, the many individual utilities are collapsed below into six scientific families. This display is explanatory only; the actual ranking used the individual diagnostics, not these family averages.


In [ ]:
utility_families = utility_family_table(candidate_scores)
utility_families.to_parquet(OUTPUT_ROOT / "candidate_utility_families.parquet", index=False)
family_columns = [
    "proposal/support",
    "LS/Fourier",
    "dispersion/smoother",
    "BLS",
    "phase repeatability",
    "event timing",
]
top_two = utility_families.loc[utility_families["baseline_rank"].le(2)].copy()
top_two["rank_label"] = top_two["baseline_rank"].map({1: "winner", 2: "runner-up"})
family_delta_rows = []
for source_id, group in top_two.groupby("source_id"):
    indexed = group.set_index("rank_label")
    if not {"winner", "runner-up"}.issubset(indexed.index):
        continue
    for family in family_columns:
        family_delta_rows.append({
            "source_id": source_id,
            "family": family,
            "winner_minus_runner_up": float(indexed.loc["winner", family] - indexed.loc["runner-up", family]),
        })
family_deltas = pd.DataFrame(family_delta_rows)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
positions = np.arange(len(family_columns))
data = [family_deltas.loc[family_deltas["family"].eq(family), "winner_minus_runner_up"].dropna() for family in family_columns]
ax.boxplot(data, positions=positions, tick_labels=family_columns, showfliers=True)
ax.axhline(0, color="0.35", linewidth=1)
ax.set_ylabel("Winner utility minus runner-up utility")
ax.set_title("Which evidence families tend to decide arbitration?")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "winner_runnerup_utility_differences.png", bbox_inches="tight")
plt.show()


## Automatically selected case studies

The casebook attempts to include an exact catalog match, a harmonic-only match, a catalog-family disagreement, an ambiguous small-margin object, and no-catalog objects with strong and weak internal evidence. Categories that do not occur in a small cohort are skipped rather than fabricated.


In [ ]:
cases = choose_case_studies(ok, per_group=CASES_PER_GROUP)
cases.to_parquet(OUTPUT_ROOT / "case_studies.parquet", index=False)
display(cases[[
    "candidate_id",
    "case_reason",
    "cohort_group",
    "selected_period_days",
    "catalog_reference_period",
    "stored_period_days",
    "selected_exact_match",
    "selected_family_match",
    "selected_score",
    "score_margin",
]])


In [ ]:
def plot_phase(ax, light_curve: pd.DataFrame, period: float, title: str, color: str) -> None:
    if not np.isfinite(period) or period <= 0:
        ax.text(0.5, 0.5, "No finite comparison period", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        return
    phase = np.mod(light_curve["JD"].to_numpy(dtype=float) / period, 1.0)
    magnitude = light_curve["mag"].to_numpy(dtype=float)
    ax.scatter(np.r_[phase, phase + 1], np.r_[magnitude, magnitude], s=7, alpha=0.35, color=color, rasterized=True)
    ax.set(xlabel="Phase", ylabel="Aligned magnitude", title=f"{title}\nP = {period:.7g} d", xlim=(0, 2))
    ax.invert_yaxis()
    ax.grid(alpha=0.15)


def plot_case_dashboard(case: pd.Series) -> Path:
    source_id = str(case["candidate_id"])
    record = cohort.loc[cohort["candidate_id"].astype(str).eq(source_id)].iloc[0]
    light_curve = load_prepared_light_curve(record)
    scores = candidate_scores.loc[candidate_scores["source_id"].astype(str).eq(source_id)].copy()
    scores = scores.sort_values("baseline_rank")
    utilities = utility_families.loc[utility_families["source_id"].astype(str).eq(source_id)].copy()
    utilities = utilities.sort_values("baseline_rank").head(6)

    selected_period = float(case["selected_period_days"])
    reference_period = pd.to_numeric(pd.Series([case["catalog_reference_period"]]), errors="coerce").iloc[0]
    runner_up_period = float(case["runner_up_period_days"])
    comparison_period = reference_period if np.isfinite(reference_period) else runner_up_period
    comparison_title = "External reference fold" if np.isfinite(reference_period) else "Runner-up fold"

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes[0, 0].scatter(light_curve["JD"], light_curve["mag"], s=7, alpha=0.45, color="#4c78a8", rasterized=True)
    axes[0, 0].invert_yaxis()
    axes[0, 0].set(xlabel="JD", ylabel="Aligned magnitude", title="Prepared ASAS-SN light curve")
    axes[0, 0].grid(alpha=0.15)

    plot_phase(axes[0, 1], light_curve, selected_period, "Selected-period fold", "#59a14f")
    plot_phase(axes[0, 2], light_curve, float(comparison_period), comparison_title, "#f28e2b")

    period_values = pd.to_numeric(scores["period_days"], errors="coerce")
    score_values = pd.to_numeric(scores["baseline_score"], errors="coerce")
    stage_colors = scores["candidate_stage"].map({"scored_shortlist": "#4c78a8", "local_refinement": "#b279a2"}).fillna("#bab0ab")
    axes[1, 0].scatter(period_values, score_values, c=stage_colors, s=18, alpha=0.65)
    axes[1, 0].scatter([selected_period], [float(case["selected_score"])], color="#e15759", marker="*", s=180, zorder=4, label="selected")
    if np.isfinite(reference_period):
        axes[1, 0].axvline(reference_period, color="#f28e2b", linestyle="--", label="catalog")
    stored_period = pd.to_numeric(pd.Series([case["stored_period_days"]]), errors="coerce").iloc[0]
    if np.isfinite(stored_period):
        axes[1, 0].axvline(stored_period, color="0.35", linestyle=":", label="stored")
    axes[1, 0].set(xscale="log", xlabel="Candidate period (d)", ylabel="Deterministic score", title="Final arbitration pool")
    axes[1, 0].legend(frameon=False, fontsize=8)
    axes[1, 0].grid(alpha=0.15, which="both")

    matrix = utilities[family_columns].to_numpy(dtype=float)
    image = axes[1, 1].imshow(matrix, aspect="auto", vmin=0, vmax=1, cmap="viridis")
    axes[1, 1].set_xticks(np.arange(len(family_columns)), family_columns, rotation=35, ha="right")
    axes[1, 1].set_yticks(np.arange(len(utilities)), [f"rank {int(value)}" for value in utilities["baseline_rank"]])
    axes[1, 1].set_title("Utility-family anatomy of top candidates")
    fig.colorbar(image, ax=axes[1, 1], label="Mean within-source utility")

    axes[1, 2].axis("off")
    info = [
        f"Case: {case['case_reason']}",
        f"Source: {source_id}",
        f"Selected: {selected_period:.8g} d",
        f"Runner-up: {runner_up_period:.8g} d",
        f"Catalog: {reference_period:.8g} d" if np.isfinite(reference_period) else "Catalog: none",
        f"Stored: {stored_period:.8g} d" if np.isfinite(stored_period) else "Stored: none",
        f"Score: {float(case['selected_score']):.3f}",
        f"Margin: {float(case['score_margin']):.3f}",
        f"Method families: {float(case['selected_independent_method_family_count']):.0f}",
        f"Exact catalog match: {case['selected_exact_match']}",
        f"Family catalog match: {case['selected_family_match']}",
    ]
    axes[1, 2].text(0.02, 0.98, "\n".join(info), va="top", ha="left", family="monospace")

    fig.suptitle(f"Period arbitration case study: {source_id}", y=1.01)
    fig.tight_layout()
    output = FIGURE_ROOT / f"case_{source_id.replace(':', '_')}.png"
    fig.savefig(output, bbox_inches="tight")
    plt.show()
    return output


case_figure_paths = [plot_case_dashboard(case) for _, case in cases.iterrows()]
print(f"Saved {len(case_figure_paths)} case dashboards to {FIGURE_ROOT}")


## Interpretation checklist

- **Exact catalog agreement** tests whether the selected period uses the same convention as the external reference.
- **Harmonic-family agreement** treats common half-, double-, and higher-multiple conventions as related solutions.
- **Candidate-oracle recovery** measures whether the search generated a usable candidate; it does not measure whether the ranker selected it.
- **Top-K recovery** shows how far down the deterministic ranking one must inspect to find a catalog-matching candidate.
- For objects **without external periods**, score margin, method support, phase-template Q, and agreement with a stored internal period are diagnostics—not ground-truth accuracy.
- The deterministic score is a within-light-curve relative ranking. It is not a false-alarm probability or a calibrated probability of periodicity, and the selector does not abstain when a candidate pool exists.

Persistent outputs include `best_periods.parquet`, `best_periods.csv`, the complete `candidate_scores.parquet`, raw `proposals.parquet`, stage and search-health tables, a configuration/commit manifest, resumable per-object checkpoints, and all figures in the `figures/` directory.


In [ ]:
output_inventory = pd.DataFrame([
    {
        "path": str(path.relative_to(OUTPUT_ROOT)),
        "size_mb": path.stat().st_size / 1024**2,
    }
    for path in sorted(OUTPUT_ROOT.rglob("*"))
    if path.is_file()
])
display(output_inventory)
print("Output root:", OUTPUT_ROOT)
